### 🛠️ Environment Setup & Dependency Verification
This notebook includes verified dependencies to ensure reproducible execution.

- **Automatic Setup:** Cell 2 verifies Python version compatibility and installs verified package versions sequentially.
- **Network Notice:** Active internet access is required to download uncached packages.


In [ ]:
# =====================================================================
# VERIFIED ENVIRONMENT DEPENDENCIES (2026-09-04 00:23:02)
# =====================================================================

import sys
import subprocess
import tempfile
import importlib.metadata

REQUIRED_PYTHON = (3, 12)
CURRENT_PYTHON = (sys.version_info.major, sys.version_info.minor)

# Major version mismatch -> Clean hard stop
if CURRENT_PYTHON[0] != REQUIRED_PYTHON[0]:
    req_major = REQUIRED_PYTHON[0]
    curr_major = CURRENT_PYTHON[0]
    print(f"❌ Error: Major Python version mismatch!")
    print(f"This notebook requires Python {req_major}.x, but your environment is running Python {curr_major}.x.\n")
    sys.exit("Execution stopped due to Python major version incompatibility.")

# Minor version mismatch -> Non-blocking warning
if CURRENT_PYTHON[1] != REQUIRED_PYTHON[1]:
    req_ver = f"{REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}"
    curr_ver = f"{CURRENT_PYTHON[0]}.{CURRENT_PYTHON[1]}"
    print(f"⚠️ This code was created with Python {req_ver}. You are trying to run it with {curr_ver}.")
    print(f"If installation fails, consider changing your runtime Python version back to {req_ver}.\n")

# Dependency Specification with Scoped Flags
DEPENDENCIES = [{'name': 'numpy', 'version': '2.0.2', 'flags': []}, {'name': 'opencv-python-headless', 'version': '4.13.0.92', 'flags': []}, {'name': 'scikit-learn', 'version': '1.6.1', 'flags': []}, {'name': 'PyYAML', 'version': '6.0.3', 'flags': []}, {'name': 'pillow', 'version': '11.3.0', 'flags': []}, {'name': 'beautifulsoup4', 'version': '4.13.5', 'flags': []}, {'name': 'umap-learn[plot]', 'version': '0.5.12', 'flags': []}]

# Informational notes & uninstalled fallbacks:
# cupy (optional or conditional dependency inside try/except block)
# this_package_does_not_exist_xyz (optional or conditional dependency inside try/except block)
# torch (imported as 'torch', not currently found in active env)

print(f"Applying verified environment dependencies [2026-09-04 00:23:02]...")
print("💡 Note: Dependencies are installed sequentially to prevent index conflicts.\n")

passed_count = 0
failed_packages = []
total_deps = len(DEPENDENCIES)
installed_baseline = {}

for idx, item in enumerate(DEPENDENCIES, start=1):
    name = item["name"]
    ver = item.get("version", "")
    flags = item.get("flags", [])
    specifier = f"{name}=={ver}" if ver else name

    # Step 1: Pre-install inspection
    # Avoids redundant re-installations in pre-configured platforms (Colab, Kaggle)
    already_satisfied = False
    try:
        current_ver = importlib.metadata.version(name)
        if not ver or current_ver == ver:
            already_satisfied = True
            passed_count += 1
            installed_baseline[name] = current_ver
            print(f"[{idx}/{total_deps}] ⚡ {name} ({current_ver}) already satisfied in environment")
    except Exception:
        pass

    if already_satisfied:
        continue

    # Step 2: Non-blocking installation via disk-backed stream redirection
    # Flags explanation:
    # - "--no-input": Prevents pip from prompting on stdin
    # - "--disable-pip-version-check": Eliminates overhead checking for newer pip releases
    # - "--no-warn-script-location": Suppresses path warnings for local bin paths
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-input",
        "--disable-pip-version-check",
        "--no-warn-script-location",
        specifier
    ] + flags

    print(f"[{idx}/{total_deps}] 📦 Installing {specifier}...")
    sys.stdout.flush()

    captured_output = []
    returncode = 0

    try:
        with tempfile.TemporaryFile(mode="w+", encoding="utf-8", errors="replace") as tmp_out:
            proc = subprocess.run(
                cmd,
                stdin=subprocess.DEVNULL,
                stdout=tmp_out,
                stderr=subprocess.STDOUT,
                timeout=120
            )
            returncode = proc.returncode
            tmp_out.seek(0)
            raw_text = tmp_out.read()
            for line in raw_text.splitlines():
                if line.strip():
                    captured_output.append(line)
                    print(f"    {line}")
            sys.stdout.flush()
    except subprocess.TimeoutExpired:
        returncode = -1
        timeout_msg = "Error: Subprocess installation exceeded per-package timeout limit (120s)."
        captured_output.append(timeout_msg)
        print("    ❌ Installation timed out after 120s.")
        sys.stdout.flush()
    except Exception as exc:
        returncode = -1
        err_msg = f"Execution failed: {exc}"
        captured_output.append(err_msg)
        print(f"    ❌ {err_msg}")
        sys.stdout.flush()

    if returncode == 0:
        passed_count += 1
        print(f"    ✅ {specifier} installed successfully")
        
        # Real-time drift audit across previously installed dependencies
        try:
            current_ver = importlib.metadata.version(name)
            installed_baseline[name] = current_ver
        except Exception:
            pass

        for prev_pkg, prev_ver in list(installed_baseline.items()):
            if prev_pkg == name:
                continue
            try:
                active_now = importlib.metadata.version(prev_pkg)
                if active_now != prev_ver:
                    print(f"   ⚠️ Dependency Drift: Installing '{specifier}' caused '{prev_pkg}' to drift from {prev_ver} ➔ {active_now}")
                    installed_baseline[prev_pkg] = active_now
            except Exception:
                pass
    else:
        err_snippet = captured_output[-1] if captured_output else "Unknown pip error"
        failed_packages.append((specifier, ver, flags, "\n".join(captured_output)))
        print(f"    ❌ {specifier} failed to install (exit code {returncode})")
        print(f"       ├─ Author Verified Version: {ver or 'unspecified'}")
        if flags:
            print(f"       ├─ Scoped Flags: {' '.join(flags)}")
        print(f"       └─ Error: {err_snippet}\n")

print("\n" + "=" * 60)
if not failed_packages:
    print(f"✅ Setup complete! All {passed_count}/{total_deps} dependencies verified.")
else:
    print(f"⚠️ Setup completed with issues: {passed_count}/{total_deps} packages installed.")
    print("Troubleshooting Steps:")
    print("1. Internet Access: Ensure your notebook environment has active internet access.")
    print("2. Unpinned Installs: Test installing failed libraries manually: '!pip install <pkg>'")
    print(f"3. Troubleshooting Steps: For a detailed guide on resolving setup errors, see: https://github.com/flyinacres/notebook_env/blob/main/HELP.md")
print("=" * 60)


In [ ]:
# --- Standard cases ---
import os                          # stdlib, should be filtered out
import numpy as np                 # aliased import, common case
import numpy                       # duplicate of above, unaliased, should collapse to one entry
from collections import OrderedDict  # stdlib from-import


In [ ]:
# --- Name mismatches (import name != pip name) ---
import cv2                         # opencv-python
import sklearn                     # scikit-learn
from sklearn.ensemble import RandomForestClassifier  # submodule from-import, should resolve to 'sklearn' root
import yaml                        # PyYAML
from PIL import Image              # pillow
import bs4                         # beautifulsoup4


In [ ]:
# --- Submodule / dotted imports ---
import xml.etree.ElementTree as ET   # stdlib, dotted, should filter to 'xml' and be dropped
import torch.nn as nn                # third-party, dotted, should resolve to 'torch' root


In [ ]:
# --- Multiple imports on one line ---
import json, re, itertools           # all stdlib, comma-separated


In [ ]:
# --- Conditional / defensive imports ---
try:
    import cupy as cp                # may not be installed, should be caught as missing if absent
except ImportError:
    cp = None


In [ ]:
# --- Imports inside a function (not at module level) ---
def load_extra():
    import umap.plot                 # nested import, still valid Python, ast.walk should still find it
    return umap.plot


In [ ]:
# --- Star import ---
from math import *                   # stdlib, star import, no explicit names to extract


In [ ]:
# --- Dynamic import (won't be caught by ast.Import/ImportFrom at all) ---
import importlib
some_module = importlib.import_module("json")   # dynamic, string-based, static AST scan will miss this


In [ ]:
# --- Import that looks like code but is commented out ---
# import nonexistent_fake_package    # should NOT be picked up, it's a comment


In [ ]:
# --- Import inside a string (should NOT be picked up) ---
example_code_snippet = "import fake_package_in_a_string" 


In [ ]:
# --- Deliberately missing / fake package ---
try:
    import this_package_does_not_exist_xyz # should be flagged as missing by find_spec
except ImportError:
    pass


In [ ]:
# --- Combined scenario: hardware tag WITH a matching index URL already in the notebook ---
# torch is already imported above (import torch.nn as nn) and will resolve to a
# +cu121 tagged version in the mocked environment. This line provides the index
# URL for it in the same notebook, which should suppress the "no download link
# found" warning and should get preserved as a --extra-index-url manifest line.
!pip install torch==2.3.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121


In [ ]:
# --- Relative import (bare `from . import x`) ---
# node.module is None for this exact form, so visit_ImportFrom's `if node.module:`
# check skips it entirely. It should be completely invisible: not extracted, not
# filtered as stdlib, not flagged as missing. Just absent, silently.
from . import helper_module
